In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from tensorflow.keras.layers import MaxPooling1D, Flatten, Dense
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from pandas.core.arrays import categorical
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
import zipfile
from tensorflow.keras.layers import Input, Conv1D
# from catboost import CatBoostClassifier
from google.colab import files
import lightgbm as lgb
import os
# !pip install catboost
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from keras.models import Sequential
from tensorflow.keras import layers
from sklearn.model_selection import RandomizedSearchCV
from keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Activation, BatchNormalization
from keras.layers import Input
# السطر الجديد اللي هيحل الأزمة:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import re
from scipy.interpolate import interp1d
# أولاً: تسطيب المكتبة الخاصة ببيانات ناسا
!pip install lightkurve --quiet
import matplotlib.pyplot as plt
import lightkurve as lk

In [ ]:
def process_stellar_data(lc):
    # 1. المرحلة الأولى: المكواة (flatten)
    lc_flat = lc.flatten(window_length=350)
        pg = lc_flat.to_periodogram(method='bls')
    best_p = pg.period_at_max_power

    # 3. المرحلة الثالثة: الطيّ (Folding) - دلوقتي نستخدم best_p وإحنا مطمنين
    lc_folded = lc_flat.fold(period=best_p)
    data = lc_folded.flux.value
    clean_data = np.where(np.isnan(data), np.nanmean(data), data)
    final_data = np.resize(clean_data, (1000,))
    return final_data

In [ ]:
model = models.Sequential()
model.add(Input(shape=(1000, 1)))
# العدسة المكبرة هنا بتمشي في اتجاه واحد (الزمن)
model.add(Conv1D(32, 3, activation='relu'))

def create_model():
    model = Sequential([
        # طبقة الإدخال: بنقول للموديل توقع 1000 نقطة
        Input(shape=(1000, 1)),

        # لازم Conv1D تكون هي الأولى عشان تفهم الأبعاد
        Conv1D(filters=32, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),

        Conv1D(filters=64, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),

        Flatten(),

        Dense(64, activation='relu'),
        Dropout(0.5),

        # النتيجة النهائية
        Dense(1, activation='sigmoid')
    ])

    return model

# 1. بناء الموديل
model = create_model()

# 2. الكومبايل (الضبط)
from tensorflow.keras.optimizers import Adam
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

print("✅ الموديل اتصلح وبقى جاهز للطيران!")

In [ ]:
confirmed_planets = [
    "Kepler-10", "Kepler-90", "Kepler-11", "Kepler-186", "Kepler-22",
    "Kepler-452", "Kepler-62", "Kepler-10b", "Kepler-7b", "Kepler-8b",
    "Kepler-12b", "Kepler-13b", "Kepler-14b", "Kepler-15b", "Kepler-16b",
    "Kepler-17b", "Kepler-18b", "Kepler-19b", "Kepler-20b", "Kepler-21b"
]

noise_stars = [
    "KIC 8462852", "KIC 5522786", "KIC 9832227", "KIC 10227020", "KIC 11442793",
    "KIC 3427344", "KIC 2157356", "KIC 10187017", "KIC 10001893", "KIC 5807616",
    "KIC 11026764", "KIC 3542116", "KIC 2436324", "KIC 4544587", "KIC 5613330",
    "KIC 6442183", "KIC 7289100", "KIC 8106973", "KIC 9025370", "KIC 11853905"
]

all_stars_features = []
all_stars_labels = []
train_stars = confirmed_planets + noise_stars

print("🎯 بدأنا شحن كتائب الـ 40 نجم (النسخة العبقرية)...")

for i, name in enumerate(train_stars):
    try:
        search = lk.search_lightcurve(name, author="Kepler", cadence="long")
        if len(search) > 0:
            lc = search[0].download()
            clean_array = process_stellar_data(lc)
            min_v, max_v = np.min(clean_array), np.max(clean_array)
            scaled_array = (clean_array - min_v) / (max_v - min_v)

            all_stars_features.append(scaled_array)
            all_stars_labels.append(1 if i < len(confirmed_planets) else 0)
    except:
        continue

X_train = np.array(all_stars_features).reshape(-1, 1000, 1)
y_train = np.array(all_stars_labels)

model = create_model()
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

print(f"💪 الميزان اتظبط: {len(X_train)} نجم جاهزين للتدريب..")
model.fit(X_train, y_train, epochs=150, batch_size=4)

test_name = "KIC 10593626"
search = lk.search_lightcurve(test_name, author="Kepler", cadence="long")
lc = search[0].download()
clean_data = process_stellar_data(lc)

min_val, max_val = np.min(clean_data), np.max(clean_data)
standard_data = (clean_data - min_val) / (max_val - min_val)
pred = model.predict(standard_data.reshape(1, 1000, 1))
print(f"\n🌟 النجم {test_name} -> نسبة الكوكب عند كوبر: {pred[0][0]*100:.2f}%")

In [ ]:
target_list = [
    "KIC 10255705", "KIC 10454113", "KIC 10905239", "KIC 11131420", "KIC 11403044",
    "KIC 11551430", "KIC 11709124", "KIC 11812732", "KIC 12156549", "KIC 12253106",
    "KIC 3116911",  "KIC 3247390",  "KIC 3342485",  "KIC 3427335",  "KIC 3455760",
    "KIC 4047514",  "KIC 4141376",  "KIC 4357062",  "KIC 4458086",  "KIC 4544587"
]
candidates = []

print("🚀 كوبر بدأ عملية المسح الشامل...")

for star in target_list:
    try:
        # 1. تحميل وتجهيز
        search = lk.search_lightcurve(star, author="Kepler", cadence="long")
        lc = search[0].download().flatten()

        # 2. البحث عن أقوى تردد (Period)
        pg = lc.to_periodogram(method='bls', period=np.arange(0.5, 10, 0.1))
        max_power = pg.max_power
        best_p = pg.period_at_max_power

        print(f"✨ فحص {star}.. قوة الإشارة: {max_power:.2f}")

        snr = max_power / np.median(pg.power)
        if max_power > 50 and snr > 10:
                candidates.append({'star': star, 'period': best_p, 'power': max_power})

    except:
        print(f"❌ تعذر تحميل بيانات {star}")

print("\n🎯 القائمة النهائية للمشتبه فيهم:")
for c in candidates:
    print(f"🌟 النجم {c['star']} عنده إشارة قوية جداً! الـ Period: {c['period']:.2f}")

In [ ]:
target_star = "KIC 10905239"
search = lk.search_lightcurve(target_star, author="Kepler", cadence="long")
lc_raw = search[0].download()
lc_processed = lc_raw.flatten().fold(period=3.50).bin(time_bin_size=0.05)
plt.figure(figsize=(10,6))
lc_processed.plot(color='skyblue', lw=4, marker='o', label='Filtered Transit')
plt.title(f"Clear Transit Signal: {target_star}\nDetected by Merv & Cooper", fontsize=14)
plt.xlabel("Phase (Time relative to transit)", fontsize=12)
plt.ylabel("Relative Brightness", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

In [ ]:
import numpy as np
from lightkurve import periodogram

# كوبر بيبحث عن التردد الصحيح للدوران
pg = lc_raw.flatten().to_periodogram(method='bls', period=np.arange(0.5, 15, 0.01))
max_power2 = pg.max_power
best_period = pg.period_at_max_power
print(f"🕵️ كوبر لقى إن الـ Period الصح هي: {best_period}")

# دلوقتي نرسم بالرقم اللي كوبر لقاه
lc_raw.flatten().fold(period=best_period).bin(time_bin_size=0.02).plot(color='skyblue', lw=2)
plt.title(f"The Truth of  KIC 10905239 | Period: {best_period}")
plt.show()

In [ ]:
depth_percent = (max_power2) * 100

if depth_percent > 2.0:
    print("⚠️ ده غالباً 'نجم ثنائي' (Eclipsing Binary) لأن الحفرة عميقة جداً.")
    star_type = "Binary Star"

elif 0.1 < depth_percent <= 1.0:
    print("🪐 ده 'عملاق غازي' محتمل (Gas Giant) زي المشتري.")
    star_type = "Gas Giant"

elif 0.001 <= depth_percent <= 0.1:
    print("🌍 مبروك! ده 'كوكب صخري' محتمل (Terrestrial) زي الأرض.")
    star_type = "Earth-like"

else:
    print("🔍 إشارة ضعيفة جداً أو مجرد ضجيج (Noise).")
    star_type = "Unknown"

In [ ]:
# # --- قائمة نجوم مرشحة (لسه مفيش حد أكد إنها كواكب 100%) ---
# # الأسماء دي من أرشيف ناسا وحالتها "Candidate"
# # --- الكتيبة الـ 20: نجوم مرشحة تحت الاختبار ---
# mass_scan_list = [
#     "KIC 3116911", "KIC 8311099", "KIC 10386984", "KIC 6444896", "KIC 5522786",
#     "KIC 11442793", "KIC 10593626", "KIC 7599132", "KIC 8242393", "KIC 8359423",
#     "KIC 9475731", "KIC 9592855", "KIC 11242721", "KIC 11764567", "KIC 12156549",
#     "KIC 10854497", "KIC 11084727", "KIC 11187424", "KIC 11299299", "KIC 11551430"
# ]

# print("🚀 انطلاق عملية المسح الكبرى.. كوبر في طريقه لأعماق المجرة!")
# discoveries = []

# for name in mass_scan_list:
#     try:
#         # البحث بمرونة: لو منفعش كبلر، جربي الكتالوج العام
#         search = lk.search_lightcurve(name, author="Kepler", cadence="long")
#         if len(search) == 0:
#             continue

#         lc = search[0].download().flatten()

#         # معالجة وتوقع
#         clean_data = process_stellar_data(lc)
#         min_v, max_v = np.min(clean_data), np.max(clean_data)
#         scaled_data = (clean_data - min_v) / (max_v - min_v)

#         pred = model.predict(scaled_data.reshape(1, 1000, 1), verbose=0)
#         prob = pred[0][0] * 100

#         if prob > 70: # هنركز بس على اللي فوق الـ 70%
#             print(f"✅ هدف محتمل: {name} | النسبة: {prob:.2f}%")
#             discoveries.append((name, prob, lc))
#         else:
#             print(f"⚪ تخطي {name} (نسبة ضعيفة: {prob:.2f}%)")

#     except Exception as e:
#         # لو النجم مش موجود أو فيه مشكلة في البيانات بنعديه بهدوء
#         continue

# print("\n--- 🎊 انتهى المسح! ---")
# if len(discoveries) > 0:
#     print(f"يا ميرف! كوبر لقى {len(discoveries)} نجوم مشكوك فيهم بقوة!")
#     print("دلوقتي دورنا نفصصهم بالرسم واحد واحد.")
# else:
#     print("🔭 ملقيناش حاجة في القايمة دي، بس ولا يهمك، المجرة فيها مليار نجم غيرهم!")

In [ ]:
lc_processed = lc_raw.flatten().fold(period=0.6100000000000001).bin(time_bin_size=0.05)

plt.figure(figsize=(10,6))
lc_processed.plot(color='skyblue', lw=4, marker='o', label='Filtered Transit')
# search[0].download(): البحث بيرجع لنا قايمة ملفات، إحنا بناخد أول ملف [0] ونعمله download عشان يبدأ يدخل في ذاكرة البرنامج ونقدر نشتغل عليه.
plt.title(f"Clear Transit Signal: {target_star}\nDetected by Merv & Cooper", fontsize=14)
plt.xlabel("Phase (Time relative to transit)", fontsize=12)
plt.ylabel("Relative Brightness", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

In [ ]:
pg = lc_raw.flatten().to_periodogram(method='bls', period=np.arange(0.5, 10, 0.01))
best_period = pg.period_at_max_power
print(f"🕵️ كوبر لقى إن الـ Period الصح هي: {best_period}")
lc_raw.flatten().fold(period=best_period).bin(time_bin_size=0.02).plot(color='lime', lw=2)
plt.title(f"The Truth of KIC 9592855 | Period: {best_period}")
plt.show()